# Walkthroughs and Exercises for Statistical Modeling and Inference with Python

*Dr. Chester Ismay*


In [ ]:
%pip install -q numpy pandas scipy matplotlib seaborn statsmodels scikit-learn

In [ ]:
# Colab setup: download the course data files into this runtime so the
# read_csv / read_excel calls below work. Runs only on Google Colab; it is a
# no-op locally or in JupyterLite, where the data files are already present.
import os, sys, urllib.request

if "google.colab" in sys.modules:
    _DATA_BASE = "https://raw.githubusercontent.com/ismayc/oreilly-statistical-modeling-and-inference-with-python/main/"
    for _fname in ["imdb_movie_sample.csv", "spotify_sample.csv"]:
        if not os.path.exists(_fname):
            print(f"Downloading {_fname} ...")
            urllib.request.urlretrieve(_DATA_BASE + _fname, _fname)

In [ ]:
import pandas as pd
import numpy as np

# Display all columns
pd.set_option('display.max_columns', None)

# Turn off scientific notation
np.set_printoptions(suppress=True)
pd.options.display.float_format = '{:.10f}'.format

# Week 1

## Walkthrough 1.1: Getting Started

### Setting Up the Python Environment

If you haven't already installed Python, Jupyter, and the necessary packages, there are instructions on the course repo in the README to do so [here](https://github.com/ismayc/oreilly-statistical-modeling-and-inference-with-python/blob/main/README.md).

In [ ]:
# Importing libraries/modules and aliasing them as needed
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import sklearn

### Load a dataset

In [ ]:
# Load in the dataset
spotify_sample = pd.read_csv("spotify_sample.csv")

### Prepare data

In [ ]:
# Select a single feature and target variable
X = spotify_sample[['acousticness']]
y = spotify_sample['energy']

### Create the model

In [ ]:
from sklearn.linear_model import LinearRegression

# Create and fit the model
model = LinearRegression()
model.fit(X, y)

# Acousticness coefficient and intercept
print("Intercept:", model.intercept_)
print("Acousticness coefficient:", model.coef_)

### Make predictions

In [ ]:
# Store predicted values for each value of X
y_pred = model.predict(X)

# Define a new data point for prediction
new_data_point = pd.DataFrame({'acousticness': [0.35]})

# Make prediction
predicted_energy = model.predict(new_data_point)

print(f"Predicted energy for acousticness =",
      f"{new_data_point.iloc[0, 0]}: {round(predicted_energy[0], 2)}")

### Visualize the regression line

In [ ]:
# Plotting the results
sns.scatterplot(x=X.acousticness, y=y, color='blue',
                label='Actual Data');
sns.lineplot(x=X.acousticness, y=y_pred, color='red',
             label='Regression Line Prediction', linewidth=2);
plt.xlabel('Acousticness');
plt.ylabel('Energy');
plt.title('Actual and Predicted Energy');
plt.tight_layout();
plt.legend();
plt.show();

### Check if assumptions of linear regression met with visual tools

> **Common Pitfall:** Do not stop at the fitted line and its coefficient. A clear fan or funnel shape in the residuals vs predicted plot signals non-constant variance (heteroscedasticity), and a curved Q-Q plot signals non-normal residuals. Ignoring these diagnostics can make the slope estimate and any inference built on it misleading even when the line looks plausible.

In [ ]:
# Check for Homoscedasticity
residuals = y - y_pred
plt.figure(figsize=(10, 5))
sns.scatterplot(x=y_pred, y=residuals, alpha=0.5)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Predicted Values')
plt.tight_layout()
plt.show();

In [ ]:
# Check for Normality of Residuals
plt.figure(figsize=(10, 5))
sns.histplot(residuals, kde=True)
plt.xlabel('Residuals')
plt.title('Histogram of Residuals')
plt.tight_layout()
plt.show();

In [ ]:
sm.qqplot(residuals, line='s')
plt.title('Q-Q Plot of Residuals')
plt.tight_layout()
plt.show();

## Exercise 1.1: Getting Started

### Setting Up the Python Environment

If you ran the `# Importing libraries and aliasing them` code above, you should
be good to proceed here. If not, scroll up and run it.

### Load a dataset

In [ ]:
# Load in the dataset
imdb_movie_sample = pd.read_csv("imdb_movie_sample.csv")

### Prepare data

In [ ]:
# Select a single feature and target variable
X = imdb_movie_sample[['votes']]
y = imdb_movie_sample['rating']

### Create the model

In [ ]:
# Create and fit the model
model = LinearRegression()
model.fit(X, y)

In [ ]:
# Votes coefficient and intercept
print("Intercept:", model.intercept_)
print("Votes coefficient:", model.coef_)

### Make predictions

In [ ]:
# Store predicted values for each value of X
y_pred = model.predict(X)

In [ ]:
# Define a new data point for prediction (200,000 votes)
new_imdb_data = pd.DataFrame({'votes': [200_000]})

In [ ]:
# Make prediction
predicted_rating = model.predict(new_imdb_data)

In [ ]:
# Output results
print(f"Predicted rating for votes = ",
      f"{new_imdb_data.iloc[0, 0]}: {round(predicted_rating[0], 2)}")

### Visualize the regression line

In [ ]:
# Plotting the results
plt.figure(figsize=(10, 7));
sns.scatterplot(x=X.votes, y=y, color='blue', label='Actual', alpha=0.4);
sns.lineplot(x=X.votes, y=y_pred, color='red', linewidth=2, label='Regression Line Prediction');
plt.xlabel('Votes');
plt.ylabel('Rating');
plt.title('Actual vs Predicted Rating');
 # Disable scientific notation on both axes
plt.ticklabel_format(style='plain', axis='both');
plt.legend();
plt.tight_layout()
plt.show();

### Check if assumptions of linear regression met with visual tools

In [ ]:
# Check for Homoscedasticity
residuals = y - y_pred
plt.figure(figsize=(10, 5))
sns.scatterplot(x=y_pred, y=residuals)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Predicted Values')
plt.tight_layout()
plt.show();

In [ ]:
# Check for Normality of Residuals
plt.figure(figsize=(10, 5))
sns.histplot(residuals, kde=True)
plt.xlabel('Residuals')
plt.title('Histogram of Residuals')
plt.tight_layout()
plt.show();

sm.qqplot(residuals, line='s')
plt.title('Q-Q Plot of Residuals')
plt.tight_layout()
plt.show();

## Walkthrough 1.2: Correlation

### Correlation matrix

In [ ]:
# Select only numeric columns
numeric_columns = spotify_sample.select_dtypes(include=[np.number])

# Calculate Pearson correlation matrix
correlation_matrix = numeric_columns.corr()

# Display the correlation matrix
correlation_matrix

### Visualizing correlation matrix

In [ ]:
# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix Heatmap')
plt.tight_layout()
plt.show();

### Visualizing relationships

In [ ]:
# Plot pairplot to visualize relationships
sns.pairplot(
  numeric_columns[['danceability', 'energy', 'loudness',
                   'valence', 'popularity']]
);
plt.suptitle('Pair Plot of Selected Features', y=1)
plt.tight_layout()
plt.show();

## Exercise 1.2: Correlation

### Correlation matrix

In [ ]:
# Select only numeric columns
numeric_columns = imdb_movie_sample.select_dtypes(include=[np.number])

# Calculate Pearson correlation matrix
correlation_matrix = numeric_columns.corr()

# Display the correlation matrix
correlation_matrix

## Visualizing correlation matrix

In [ ]:
# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix Heatmap')
plt.tight_layout()
plt.show();

### Visualizing relationships

In [ ]:
# Plot pairplot to visualize relationships of
# runtime_in_minutes, rating, votes, and gross_in_dollars
sns.pairplot(
  numeric_columns[['runtime_in_minutes', 'rating', 'votes', 'gross_in_dollars']]
);
plt.suptitle('Pair Plot of Selected Features', y=1)
plt.tight_layout()
plt.show();

## Walkthrough 1.3: Multiple Regression

> **Common Pitfall:** Each coefficient in a multiple regression is the effect of that predictor holding the others fixed, not its standalone effect. When predictors are correlated with one another (multicollinearity), individual coefficients can become unstable and hard to interpret: the sign can even flip relative to the simple one-predictor model. Always report coefficients with their units and context rather than as bare numbers.

In [ ]:
# Prepare the data for multiple regression by choosing
# Predictors: danceability, energy, and loudness
# Response: popularity
X = spotify_sample[['danceability', 'energy', 'loudness']]
y = spotify_sample['popularity']

In [ ]:
# Fit the multiple regression model
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

In [ ]:
# Coefficients and intercept
print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)

In [ ]:
# Plot predicted vs. actual
sns.scatterplot(x=y, y=y_pred, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], color='red', linestyle='--', label='Perfect Fit')
plt.xlabel('Actual Popularity')
plt.ylabel('Predicted Popularity')
plt.title('Predicted vs. Actual Popularity')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show();

## Exercise 1.3: Multiple Regression

In [ ]:
# Prepare the data for multiple regression by choosing
# Predictors: runtime_in_minutes, votes, and gross_in_dollars
# Response: rating
X = imdb_movie_sample[['runtime_in_minutes', 'votes', 'gross_in_dollars']]
y = imdb_movie_sample['rating']

In [ ]:
# Fit the model
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

In [ ]:
# Coefficients and intercept
print("Intercept:", model.intercept_)
print("Coefficients:", model.coef_)

In [ ]:
# Plot predicted vs. actual
sns.scatterplot(x=y, y=y_pred, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], color='red', linestyle='--', label='Perfect Fit')
plt.xlabel('Actual Rating')
plt.ylabel('Predicted Rating')
plt.title('Predicted vs. Actual Rating')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show();

## Walkthrough 1.4: Logistic Regression

> **Common Pitfall:** A logistic regression coefficient is a change in log-odds, not a change in probability. A positive loudness coefficient means louder tracks have higher odds of being high energy, but you cannot read the coefficient directly as "probability goes up by that amount." To talk in probabilities, use `predict_proba`, and remember the same one-unit change shifts probability differently depending on where you are on the S-curve.

### Logistic Regression with a Single Predictor

In [ ]:
from sklearn.linear_model import LogisticRegression

# Set loudness as predictor/feature and binarize energy: high energy = 1
X = spotify_sample[['loudness']]
y = (spotify_sample['energy'] > 0.6).astype(int)

In [ ]:
# Fit model
model = LogisticRegression()
model.fit(X, y)

In [ ]:
# Predict over a smooth range
x_vals = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
probs = model.predict_proba(x_vals)[:, 1]

In [ ]:
# Plot predicted and actual energy against loudness
plt.plot(x_vals, probs, color='red', label='Predicted Probability (S-curve)')
plt.scatter(X, y, alpha=0.3, color='gray', label='Actual')
plt.title('Predicting High Energy from Loudness')
plt.xlabel('Loudness')
plt.ylabel('Probability of High Energy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show();

### Logistic Regression with Multiple Predictors

In [ ]:
# Define predictors (loudness and acousticness) and binary energy response
X = spotify_sample[['loudness', 'acousticness']]
y = (spotify_sample['energy'] > 0.6).astype(int)

In [ ]:
# Fit the logistic regression model
model = LogisticRegression()
model.fit(X, y)

In [ ]:
# Predict probabilities
spotify_sample['predicted_prob'] = model.predict_proba(X)[:, 1]

In [ ]:
# Plot the predicted probabilities
plt.figure(figsize=(10, 5))
sns.histplot(spotify_sample['predicted_prob'], bins=20, kde=True)
plt.title('Predicted Probabilities of High Popularity')
plt.xlabel('Predicted Probability')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show();

In [ ]:
# Scatter plot: predicted probability vs actual outcome
plt.figure(figsize=(10, 5))
sns.scatterplot(x=spotify_sample['predicted_prob'], y=y, alpha=0.5)
plt.title('Predicted Probability vs Actual High Popularity')
plt.xlabel('Predicted Probability')
plt.ylabel('Actual (0 = Low, 1 = High)')
plt.grid(True)
plt.tight_layout()
plt.show();

## Exercise 1.4: Logistic Regression

### Logistic Regression with a Single Predictor

In [ ]:
# Define votes predictor and
# binary rating response (rating > 7 -> high rating)
X = imdb_movie_sample[['votes']]
y = (imdb_movie_sample['rating'] > 7).astype(int)

In [ ]:
# Fit model
model = LogisticRegression()
model.fit(X, y)

In [ ]:
# Predict over a smooth range
x_vals = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
probs = model.predict_proba(x_vals)[:, 1]

In [ ]:
# Plot predicted and actual energy against loudness
plt.plot(x_vals, probs, color='red', label='Predicted Probability (S-curve)')
sns.scatterplot(x=X.votes, y=y, alpha=0.3, color='gray', label='Actual')
plt.title('Predicting High Rating from Votes')
plt.xlabel('Votes')
plt.ylabel('Probability of High Rating')
plt.ticklabel_format(style='plain', axis='both');
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show();

### Logistic Regression with Multiple Predictors

In [ ]:
# Define predictors (runtime_in_minutes and votes) and
# binary rating response (rating > 7 -> high rating)
X = imdb_movie_sample[['runtime_in_minutes', 'votes']]
y = (imdb_movie_sample['rating'] > 7).astype(int)

In [ ]:
# Fit the model
model = LogisticRegression()
model.fit(X, y)

In [ ]:
# Predict probabilities
imdb_movie_sample['predicted_prob'] = model.predict_proba(X)[:, 1]

In [ ]:
# Plot the predicted probabilities
plt.figure(figsize=(10, 5))
sns.histplot(imdb_movie_sample['predicted_prob'], bins=20, kde=True)
plt.title('Predicted Probabilities of High Rating')
plt.xlabel('Predicted Probability')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show();

In [ ]:
# Scatter plot: predicted probability vs actual outcome
plt.figure(figsize=(10, 5))
sns.scatterplot(x=imdb_movie_sample['predicted_prob'], y=y, alpha=0.5)
plt.title('Predicted Probability vs Actual High Rating')
plt.xlabel('Predicted Probability')
plt.ylabel('Actual (0 = Low, 1 = High)')
plt.grid(True)
plt.tight_layout()
plt.show();

## Walkthrough 1.5: ANOVA

### One-way ANOVA

In [ ]:
from statsmodels.formula.api import ols

# Choose some genres to compare
genres = ['pop', 'rock', 'hip-hop', 'jazz']

In [ ]:
# Subset the data to focus only on these genres
spotify_sample_subset = spotify_sample[spotify_sample['track_genre'].isin(genres)]

In [ ]:
# Perform one-way ANOVA analyzing energy across different genres
model = ols('energy ~ C(track_genre)', data=spotify_sample_subset).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

### Boxplot across groups

In [ ]:
# Boxplot for visualization
plt.figure(figsize=(12, 3))
sns.boxplot(y='track_genre', x='energy', data=spotify_sample_subset)
plt.title('Energy Levels Across Different Genres')
plt.xlabel('Energy')
plt.ylabel('Genre')
plt.tight_layout()
plt.show();

### Two-way ANOVA

In [ ]:
# Perform two-way ANOVA analyzing energy across genres and explicit content
model = ols('energy ~ C(track_genre) * C(explicit)', data=spotify_sample_subset).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

### Boxplot across groups with color added

In [ ]:
# Boxplot for visualization
plt.figure(figsize=(12, 3))
sns.boxplot(x='track_genre', y='energy', hue='explicit', data=spotify_sample_subset)
plt.title('Energy Levels Across Genres and Explicit Content')
plt.xlabel('Track Genre')
plt.ylabel('Energy')
plt.xticks(rotation=90);
plt.legend(title='Explicit')
plt.tight_layout()
plt.show();

## Exercise 1.5: ANOVA

### One-way ANOVA

In [ ]:
# Choose some genres to compare ('Action', 'Crime', 'Horror', 'Romance')
movie_genres = ['Action', 'Crime', 'Horror', 'Romance']

# Subset the data to focus only on these genres
imdb_movie_sample_subset = imdb_movie_sample[imdb_movie_sample['genre'].isin(movie_genres)]

# Perform one-way ANOVA analyzing rating across different genres
model = ols('rating ~ C(genre)', data=imdb_movie_sample_subset).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

### Boxplot across groups

In [ ]:
# Boxplot for visualization
plt.figure(figsize=(12, 4))
sns.boxplot(x='genre', y='rating', data=imdb_movie_sample_subset)
plt.title('Movie Ratings Across Different Genres')
plt.xlabel('Genre')
plt.ylabel('Rating')
plt.xticks(rotation=90);
plt.tight_layout()
plt.show();

### Two-way ANOVA

In [ ]:
# Perform two-way ANOVA analyzing rating across genres and decades
model = ols('rating ~ C(genre) * C(decade)', data=imdb_movie_sample_subset).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
anova_table

### Boxplot across groups with color added

In [ ]:
# Some customization here to get the decades to appear in the appropriate order

# Define the ordered categories
decades = ['1940s', '1950s', '1960s', '1970s', '1980s', '1990s',
           '2000s', '2010s', '2020s']

# Convert 'decade' column to a categorical type with ordered categories
imdb_movie_sample_subset['decade'] = pd.Categorical(
  imdb_movie_sample_subset['decade'], categories=decades, ordered=True)

# Boxplot for visualization
plt.figure(figsize=(12, 5))
sns.boxplot(x='genre', y='rating', hue='decade', data=imdb_movie_sample_subset)
plt.title('Movie Ratings Across Genres and Decades')
plt.xlabel('Genre')
plt.ylabel('Rating')
plt.legend(title='Decade')
plt.tight_layout()
plt.show();

### Self-Check

By the end of this week, you should be able to:

- [ ] Fit a simple linear regression with scikit-learn and interpret the intercept and slope in the units of the data.
- [ ] Use a fitted model to predict a response for a new feature value, and recognize when a prediction would be an unsafe extrapolation.
- [ ] Build and read a correlation matrix and pairplot to spot relationships and possible multicollinearity among predictors.
- [ ] Fit a multiple linear regression and explain each coefficient as the effect of one predictor holding the others fixed.
- [ ] Check the homoscedasticity and normality-of-residuals assumptions using residual plots, histograms, and Q-Q plots.
- [ ] Fit a logistic regression, interpret its coefficients on the log-odds scale, and obtain predicted probabilities.
- [ ] Run one-way and two-way ANOVA to compare a numeric outcome across categorical groups and visualize the groups with boxplots.

# Week 2

## Walkthrough 2.1: Kruskal-Wallis and Mann-Whitney U Tests

### Kruskal-Wallis

In [ ]:
import scipy.stats as stats

# Perform Kruskal-Wallis Test on 'danceability' grouped by 'track_genre'
grouped_data = [group["danceability"].values
    for name, group in spotify_sample.groupby("track_genre")]
stat, p_value = stats.kruskal(*grouped_data)
print(f"Kruskal-Wallis Test Statistic: {stat}")
print(f"P-value: {p_value}")

### Mann-Whitney U

In [ ]:
# Perform Mann-Whitney U Test on 'energy' for pop versus rock
group1 = spotify_sample[spotify_sample["track_genre"] == "pop"]["energy"]
group2 = spotify_sample[spotify_sample["track_genre"] == "rock"]["energy"]
stat, p_value = stats.mannwhitneyu(group1, group2)
print(f"Mann-Whitney U Test Statistic: {stat}")
print(f"P-value: {p_value}")

## Exercise 2.1: Kruskal-Wallis and Mann-Whitney U Tests

### Kruskal-Wallis

In [ ]:
# Perform Kruskal-Wallis Test on 'rating' grouped by 'decade'
grouped_data = [group["rating"].values
                  for name, group in imdb_movie_sample.groupby("decade")]
stat, p_value = stats.kruskal(*grouped_data)
print(f"Kruskal-Wallis Test Statistic: {stat}")
print(f"P-value: {p_value}")

### Mann-Whitney U

In [ ]:
# Perform Mann-Whitney U Test on 'gross_in_dollars' for 1990s versus 2000s
group1 = imdb_movie_sample[imdb_movie_sample["decade"] == "1990s"]\
            ["gross_in_dollars"]
group2 = imdb_movie_sample[imdb_movie_sample["decade"] == "2000s"]\
            ["gross_in_dollars"]
stat, p_value = stats.mannwhitneyu(group1, group2)
print(f"Mann-Whitney U Test Statistic: {stat}")
print(f"P-value: {p_value}")

## Walkthrough 2.2: Correlation and Regression Non-parametrics

> **Common Pitfall:** A strong Spearman or Kendall correlation tells you two variables move together in rank, not that one causes the other. Correlation is not causation, a third variable or the direction of the relationship can easily be the real story. Use these rank-based measures (and Theil-Sen) when relationships are monotonic but not necessarily linear or when outliers would distort Pearson correlation.

### Spearman's Rank Correlation

In [ ]:
from scipy.stats import spearmanr, kendalltau
from sklearn.linear_model import TheilSenRegressor

# Filter numeric columns
numeric_columns = spotify_sample.select_dtypes(include=[np.number])

# Calculate Spearman's rank correlation matrix
spearman_corr = numeric_columns.corr(method='spearman')
spearman_corr

# Heatmap for Spearman's rank correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(spearman_corr, annot=True, cmap='coolwarm',
            linewidths=0.5, fmt=".2f")
plt.title('Spearman Rank Correlation Matrix')
plt.tight_layout()
plt.show();

### Kendall's Tau Correlation

In [ ]:
# Calculate Kendall's tau correlation matrix
kendall_corr = numeric_columns.corr(method='kendall')
kendall_corr

# Heatmap for Kendall's tau correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(kendall_corr, annot=True, cmap='coolwarm',
            linewidths=0.5, fmt=".2f")
plt.title('Kendall Tau Correlation Matrix')
plt.tight_layout()
plt.show();

### Theil-Sen Robust Regression

In [ ]:
# Theil-Sen robust regression for 'danceability' and 'energy'
# predicting 'popularity'
X = numeric_columns[['danceability', 'energy']]
y = numeric_columns['popularity']
theil_sen = TheilSenRegressor(random_state=2025)
theil_sen.fit(X, y)
print("Intercept:", theil_sen.intercept_)
print("Coefficients:", theil_sen.coef_)

## Exercise 2.2: Correlation and Regression Non-parametrics

### Spearman's Rank Correlation

In [ ]:
# Filter numeric columns
numeric_columns = imdb_movie_sample.select_dtypes(include=[np.number])

# Calculate Spearman's rank correlation matrix
spearman_corr = numeric_columns.corr(method='spearman')
print(spearman_corr)

# Heatmap for Spearman's rank correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(spearman_corr, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Spearman Rank Correlation Matrix')
plt.tight_layout()
plt.show();

### Kendall's Tau Correlation

In [ ]:
# Calculate Kendall's tau correlation matrix
kendall_corr = numeric_columns.corr(method='kendall')
print(kendall_corr)

# Heatmap for Kendall's tau correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(kendall_corr, annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Kendall Tau Correlation Matrix')
plt.tight_layout()
plt.show();

### Theil-Sen Robust Regression

In [ ]:
# Theil-Sen robust regression for 'votes' and 'runtime_in_minutes' predicting 'rating'
X = numeric_columns[['votes', 'runtime_in_minutes']]
y = numeric_columns['rating']
theil_sen = TheilSenRegressor(random_state=2025)
theil_sen.fit(X, y)
print("Intercept:", theil_sen.intercept_)
print("Coefficients:", theil_sen.coef_)

## Walkthrough 2.3: Bootstrapping

In [ ]:
# Function to perform bootstrapping
def bootstrap(data, n_bootstrap_samples=1000, seed=2025):
    rng = np.random.default_rng(seed)
    bootstrap_samples = []
    for _ in range(n_bootstrap_samples):
        resample = data.sample(frac=1, replace=True, random_state=rng)
        bootstrap_samples.append(resample.mean())
    return bootstrap_samples

# Perform bootstrapping on the 'energy' column
bootstrap_means = bootstrap(spotify_sample['energy'])

# Plot the distribution of bootstrap means
plt.figure(figsize=(10, 5))
plt.hist(bootstrap_means, bins=30, edgecolor='k', alpha=0.7)
plt.title('Distribution of Bootstrap Means for Energy')
plt.xlabel('Mean Energy')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show();

## Exercise 2.3: Bootstrapping

In [ ]:
# Perform bootstrapping on the 'runtime_in_minutes' column
# using the bootstrap function defined above
bootstrap_means = bootstrap(imdb_movie_sample['runtime_in_minutes'])

# Plot the distribution of bootstrap means
plt.figure(figsize=(10, 5))
plt.hist(bootstrap_means, bins=30, edgecolor='k', alpha=0.7)
plt.title('Distribution of Bootstrap Means for Runtime')
plt.xlabel('Mean Runtime (minutes)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show();

## Walkthrough 2.4: Confidence Intervals

> **Common Pitfall:** A 95% bootstrap confidence interval is a statement about the population mean, not about individual observations, and it does not mean "95% of the data fall in this range." It means the procedure captures the true mean about 95% of the time across repeated samples. Widen your sample or increase the number of resamples for a more stable interval rather than reading too much into one narrow result.

In [ ]:
# Function to calculate bootstrap confidence intervals
def bootstrap_confidence_interval(data, n_bootstrap_samples=1000, confidence_level=0.95,
                                  seed=2025):
    rng = np.random.default_rng(seed)
    bootstrap_samples = []
    for _ in range(n_bootstrap_samples):
        resample = data.sample(frac=1, replace=True, random_state=rng)
        bootstrap_samples.append(resample.mean())
    lower_bound = np.percentile(bootstrap_samples, (1 - confidence_level) / 2 * 100)
    upper_bound = np.percentile(bootstrap_samples, (1 + confidence_level) / 2 * 100)
    return lower_bound, upper_bound

# Calculate confidence intervals for the 'energy' column
lower, upper = bootstrap_confidence_interval(spotify_sample['energy'])
print(f"95% Confidence Interval for Mean Energy: [{lower}, {upper}]")

# Plot the distribution of bootstrap means and confidence interval
rng = np.random.default_rng(2025)
bootstrap_means = [spotify_sample['energy'].sample(frac=1, replace=True, random_state=rng).mean() for _ in range(1000)]
plt.figure(figsize=(10, 5))
plt.hist(bootstrap_means, bins=30, edgecolor='k', alpha=0.7)
plt.axvline(lower, color='r', linestyle='--')
plt.axvline(upper, color='r', linestyle='--')
plt.title('Bootstrap Means and 95% Confidence Interval for Energy')
plt.xlabel('Mean Energy')
plt.ylabel('Frequency')
plt.show();

## Exercise 2.4: Confidence Intervals

In [ ]:
# Calculate confidence intervals for the 'rating' column using the
# bootstrap_confidence_interval function defined above
lower, upper = bootstrap_confidence_interval(imdb_movie_sample['rating'])
print(f"95% Confidence Interval for Mean Rating: [{lower}, {upper}]")

# Plot the distribution of bootstrap means and confidence interval
rng = np.random.default_rng(2025)
bootstrap_means = [imdb_movie_sample['rating'].sample(frac=1, replace=True, random_state=rng).mean() for _ in range(1000)]
plt.figure(figsize=(10, 5))
plt.hist(bootstrap_means, bins=30, edgecolor='k', alpha=0.7)
plt.axvline(lower, color='r', linestyle='--')
plt.axvline(upper, color='r', linestyle='--')
plt.title('Bootstrap Means and 95% Confidence Interval for Rating')
plt.xlabel('Mean Rating')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show();

## Walkthrough 2.5: Real-World Scenarios for Bootstrapping

**Music Industry Analytics**

*Objective*: Analyzing song popularity and estimating population mean popularity.

*Scenario*: You are a data scientist for a movie production company. Your role is to analyze movie ratings and predict the average rating of movies produced. By using bootstrapping techniques, you aim to estimate the mean movie rating and its variability to understand the unknown population mean.

*Bootstrapping Applications*:

- *Estimating Average Popularity*: Use bootstrapping to calculate the mean and confidence intervals of the popularity scores.
- *Assessing Variability*: Determine the variability in the means of song popularity.

In [ ]:
# Calculate bootstrap statistics for the 'popularity' column

# Function to calculate bootstrap statistics
def bootstrap_statistics(data, n_bootstrap_samples=1000, seed=2025):
    rng = np.random.default_rng(seed)
    bootstrap_means = []
    for _ in range(n_bootstrap_samples):
        resample = data.sample(frac=1, replace=True, random_state=rng)
        bootstrap_means.append(resample.mean())
    return np.mean(bootstrap_means), np.std(bootstrap_means)

mean_popularity, std_popularity = bootstrap_statistics(spotify_sample['popularity'])
print(f"Bootstrapped Mean Popularity: {mean_popularity}")
print(f"Bootstrapped Std Dev Popularity: {std_popularity}")

# Plot the distribution of bootstrap means
rng = np.random.default_rng(2025)
bootstrap_means = [spotify_sample['popularity'].sample(frac=1, replace=True, random_state=rng).mean() for _ in range(1000)]
plt.figure(figsize=(10, 5))
plt.hist(bootstrap_means, bins=30, edgecolor='k', alpha=0.7)
plt.axvline(mean_popularity, color='r', linestyle='--')
plt.title('Bootstrap Means for Popularity')
plt.xlabel('Mean Popularity')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show();

## Exercise 2.5: Real-World Scenarios for Bootstrapping

**Film Industry Insights**

*Objective*: Evaluating movie gross revenue and estimating population mean gross revenue.

*Scenario*: You are a data scientist for a movie production company. Your role is to analyze movie gross revenue and predict the average gross revenue of movies produced. By using bootstrapping techniques, you aim to estimate the mean gross revenue and its variability to understand the unknown population mean.

*Bootstrapping Applications*:

- *Estimating Average Gross Revenue*: Use bootstrapping to calculate the mean and confidence intervals of movie gross revenue.
- *Assessing Variability*: Determine the variability in the means of movie gross revenue.

In [ ]:
# Calculate bootstrap statistics for the 'gross_in_dollars' column
# using the function called bootstrap_statistics defined above
mean_gross, std_gross = bootstrap_statistics(imdb_movie_sample['gross_in_dollars'])
print(f"Bootstrapped Mean Gross Revenue: {mean_gross}")
print(f"Bootstrapped Std Dev Gross Revenue: {std_gross}")

# Plot the distribution of bootstrap means
rng = np.random.default_rng(2025)
bootstrap_means = [imdb_movie_sample['gross_in_dollars'].sample(frac=1, replace=True, random_state=rng).mean() for _ in range(1000)]
plt.figure(figsize=(10, 5))
plt.hist(bootstrap_means, bins=30, edgecolor='k', alpha=0.7)
plt.axvline(mean_gross, color='r', linestyle='--')
plt.title('Bootstrap Means for Gross Revenue')
plt.xlabel('Mean Gross Revenue')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show();

### Self-Check

By the end of this week, you should be able to:

- [ ] Choose and run the Kruskal-Wallis and Mann-Whitney U tests as non-parametric alternatives when ANOVA or t-test assumptions are in doubt.
- [ ] Compute Spearman and Kendall rank correlations and explain when they are preferable to Pearson correlation.
- [ ] Fit a Theil-Sen robust regression and describe why it resists the influence of outliers.
- [ ] Write a bootstrap procedure that resamples with replacement to build a sampling distribution of a statistic.
- [ ] Construct and correctly interpret a bootstrap confidence interval for a population mean.
- [ ] Apply bootstrapping to a real-world scenario to estimate a mean and quantify its variability.